In [1]:
!pip install sqlalchemy


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    Date,
    func
)

from sqlalchemy.orm import declarative_base, sessionmaker

In [4]:
data = pd.read_csv("cleaned1_covid-19_cases_canada.csv")

print("CSV records:", len(data))

CSV records: 1461


In [5]:
data["date_report"] = pd.to_datetime(
    data["date_report"],
    format="%d-%m-%Y",
    errors="coerce"
).dt.date

data["report_week"] = pd.to_datetime(
    data["report_week"],
    format="%d-%m-%Y",
    errors="coerce"
).dt.date

In [6]:
print(data[["date_report", "report_week"]].head())

  date_report report_week
0  2020-01-25  2020-01-19
1  2020-01-27  2020-01-26
2  2020-01-28  2020-01-26
3  2020-01-31  2020-01-26
4  2020-02-04  2020-02-02


In [7]:
print("Missing date_report:", data["date_report"].isnull().sum())
print("Missing report_week:", data["report_week"].isnull().sum())

Missing date_report: 0
Missing report_week: 0


In [8]:
engine = create_engine(
    "sqlite:///covid_q2.db",
    echo=False
)

Base = declarative_base()

In [9]:
class CovidCase(Base):

    __tablename__ = "covid_cases"

    id = Column(
        Integer,
        primary_key=True,
        autoincrement=True
    )

    provincial_case_id = Column(String)
    age = Column(String)
    sex = Column(String)
    health_region = Column(String)
    province = Column(String)
    country = Column(String)

    date_report = Column(Date)
    report_week = Column(Date)

    has_travel_history = Column(String)
    locally_acquired = Column(String)
    case_source = Column(String)

In [10]:
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

print("Fresh table created.")

Fresh table created.


In [11]:
Session = sessionmaker(bind=engine)
session = Session()

In [12]:
cases = []

for _, row in data.iterrows():

    case = CovidCase(
        provincial_case_id=str(row["provincial_case_id"]),
        age=str(row["age"]),
        sex=str(row["sex"]),
        health_region=str(row["health_region"]),
        province=str(row["province"]),
        country=str(row["country"]),

        date_report=row["date_report"],
        report_week=row["report_week"],

        has_travel_history=str(row["has_travel_history"]),
        locally_acquired=str(row["locally_acquired"]),
        case_source=str(row["case_source"])
    )

    cases.append(case)

session.add_all(cases)
session.commit()

print("Data inserted successfully!")

Data inserted successfully!


In [13]:
total_records = session.query(
    func.count(CovidCase.id)
).scalar()

print("CSV records:", len(data))
print("Database records:", total_records)

CSV records: 1461
Database records: 1461


In [14]:
dates = session.query(
    CovidCase.date_report
).limit(5).all()

for row in dates:
    print(row)

(datetime.date(2020, 1, 25),)
(datetime.date(2020, 1, 27),)
(datetime.date(2020, 1, 28),)
(datetime.date(2020, 1, 31),)
(datetime.date(2020, 2, 4),)


In [15]:
result = (
    session.query(
        func.strftime(
            "%Y-%m",
            CovidCase.date_report
        ).label("Month"),

        CovidCase.sex,

        func.count(
            CovidCase.id
        ).label("Total")
    )

    .filter(
        CovidCase.sex.in_(["Male", "Female"])
    )

    .group_by(
        func.strftime(
            "%Y-%m",
            CovidCase.date_report
        ),
        CovidCase.sex
    )

    .order_by(
        func.strftime(
            "%Y-%m",
            CovidCase.date_report
        )
    )

    .all()
)

In [16]:
for row in result:
    print(row)

('2020-01', 'Female', 2)
('2020-01', 'Male', 2)
('2020-02', 'Female', 10)
('2020-02', 'Male', 6)
('2020-03', 'Female', 416)
('2020-03', 'Male', 425)
('2020-04', 'Female', 247)
('2020-04', 'Male', 192)


In [17]:
result = (
    session.query(
        CovidCase.age,
        func.count(CovidCase.id).label("Female Cases")
    )
    .filter(
        CovidCase.sex == "Female"
    )
    .group_by(
        CovidCase.age
    )
    .order_by(
        func.count(CovidCase.id).desc()
    )
    .all()
)

In [18]:
for row in result:
    print(row)

('50-59', 145)
('60-69', 106)
('30-39', 105)
('20-29', 88)
('40-49', 84)
('70-79', 68)
('80-89', 29)
('90-99', 28)
('0-19', 17)
('Oct-19', 5)


In [19]:
older_age_groups = [
    "50-59",
    "60-69",
    "70-79",
    "80-89",
    "90-99"
]

In [20]:
result = (
    session.query(
        func.strftime(
            "%Y-%m",
            CovidCase.date_report
        ).label("Month"),

        func.count(
            CovidCase.id
        ).label("Total Cases")
    )

    .filter(
        CovidCase.has_travel_history == "No"
    )

    .filter(
        CovidCase.age.in_(older_age_groups)
    )

    .group_by(
        func.strftime(
            "%Y-%m",
            CovidCase.date_report
        )
    )

    .order_by(
        func.count(
            CovidCase.id
        ).desc()
    )

    .limit(2)

    .all()
)

In [21]:
for row in result:
    print(row)

('2020-04', 128)
('2020-03', 101)
